**MAESTRÍA EN INTELIGENCIA ARTIFICIAL APLICADA**

**Curso: TC5053 - Ciencia y analítica de datos**

Tecnológico de Monterrey

Prof Grettel Barceló Alonso

**Semana 6**
Ingeniería de características (FE)

---

*   **GRUPO #1**
*   ENMANUEL ABREU - A01841472
*   LUIS ACOSTA - A01027765
*   DIANA ACOSTA - A01841464
*   EDUARDO AGUA - A01841466

En esta actividad trabajarás con el archivo `computer_prices.csv`, basado en un conjunto de datos sobre características técnicas y especificaciones de computadoras portátiles y de escritorio, disponible en Kaggle.

Los datos fueron recopilados para analizar el rendimiento y el precio de los dispositivos, e incluyen información sobre hardware, almacenamiento, conectividad y otras especificaciones técnicas. Los indicadores incluidos son:

* `device_type`: Tipo de dispositivo (ej. laptop, desktop)
* `brand`: Marca del dispositivo
* `model`: Modelo del dispositivo
* `release_year`: Año de lanzamiento del dispositivo
* `os`: Sistema operativo instalado
* `form_factor`: Factor de forma o diseño del dispositivo (ej. laptop, ultrabook, desktop tower)
* `cpu_brand`: Marca del procesador
* `cpu_tier`: Nivel o gama del procesador, ordinal del 1 al 6 según desempeño
* `cpu_cores`: Número de núcleos del procesador
* `cpu_threads`: Número de hilos de ejecución del procesador
* `gpu_brand`: Marca de la tarjeta gráfica
* `gpu_model`: Modelo específico de la tarjeta gráfica
* `gpu_tier`: Nivel o gama de la GPU, ordinal del 1 al 6 según desempeño
* `vram_gb`: Memoria de video de la GPU en gigabytes
* `ram_gb`: Memoria RAM del dispositivo en gigabytes
* `storage_type`: Tipo de almacenamiento (ej. HDD, SSD)
* `storage_gb`: Capacidad de almacenamiento en gigabytes
* `storage_drive_count`: Número de unidades de almacenamiento instaladas
* `display_type`: Tipo de pantalla (ej. IPS, TN, OLED)
* `charger_watts`: Potencia del cargador (en watts) para laptops
* `psu_watts`: Potencia de la fuente de poder (en watts) para desktops
* `wifi`: Estándar de conectividad Wi-Fi (ej. Wi-Fi 5, 6, 6E, 7)
* `bluetooth`: Versión de Bluetooth
* `weight_kg`: Peso del dispositivo en kilogramos
* `warranty_months`: Meses de garantía del dispositivo
* `price`: Precio del dispositivo. Es la variable de salida o *target*, es decir, la que se pretende predecir más adelante al construir el modelo.

**NOTA IMPORTANTE:** Asegúrate de responder *explícitamente* todos los cuestionamientos.

In [ ]:
# Instalar las bibliotecas necesarias
!pip install category_encoders

In [ ]:
# Importar las bibliotecas necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date
from scipy.stats import probplot
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.preprocessing import FunctionTransformer, PowerTransformer
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from category_encoders.binary import BinaryEncoder

1. Descarga el archivo: `computer_prices.csv` y guarda, en un dataframe (`compu_df`), todos sus registros.
* Utiliza el método `info()` del dataframe, para obtener el resumen de los tipos de datos. ¿Cuántas columnas son numéricas y cuántas de texto?
* Determina la cantidad de valores únicos por columna.
* Elimina las variables:
  * `model`: Debido a su altísima cardinalidad, lo que dificulta su uso en análisis y modelado.
  * `cpu_model`: Además de su elevada cardinalidad, su información ya está representada de manera implícita en otras variables como: `cpu_tier`, `cpu_cores` y `cpu_threads`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cargar el archivo en el DataFrame solicitado.
compu_df = pd.read_csv('/content/drive/MyDrive/Ciencia de Datos/Semana 6/computer_prices.csv')
compu_df

In [ ]:
# Revisar la estructura y los tipos de datos.
compu_df.info()

# Contar las columnas numéricas y de texto.
print("Numéricas:", len(compu_df.select_dtypes(include='number').columns))
print("Texto:", len(compu_df.select_dtypes(include='object').columns))

***Respuesta:*** De acuerdo con los datos obtenidos mediante la función `info()` tenemos:
*   **15 columnas numéricas:** 12 de tipo int64 y 3 de tipo float64.
*   **12 columnas de texto:** de tipo object.

In [ ]:
# Contar los valores diferentes de cada columna.
compu_df.nunique()

*   *Entre los resultados destacan model, con 99,036 valores únicos, y cpu_model, con 26,971. Esto confirma una elevada cardinalidad.*


In [ ]:
# Eliminar las columnas que no utilizaremos.
compu_df.drop(columns=['model', 'cpu_model'], inplace=True)

# Verificar el nuevo tamaño.
print(compu_df.shape)

*   ***Resultado**: 100,000 filas y 25 columnas.*


2. Antes de iniciar con el análisis univariado, verifica si hay valores duplicados y/o faltantes.
* Obtén las estadísticas descriptivas, separado las numéricas y las categóricas. De estas últimas incluye las tablas de frecuencia.
* Genera histogramas para las numéricas y diagramas de barras para las categóricas. Con alta cardinalidad, sólo incluye los 10 valores más frecuentes.

In [ ]:
# Contamos los registros duplicados.
print("Duplicados:", compu_df.duplicated().sum())

# Contamos los valores faltantes por columna.
print("\nValores faltantes:")
print(compu_df.isnull().sum())


*   *El archivo no presenta registros duplicados ni valores faltantes.*


In [ ]:
# Estadísticas de las variables numéricas.
compu_df.describe().T

In [ ]:
# Estadísticas de las variables categóricas.
compu_df.describe(include=['object','category']).T

In [ ]:
# Tablas de frecuencia de cada variable categórica.
num_cols = compu_df.select_dtypes(include=['number']).columns
cat_cols = compu_df.select_dtypes(include=['category','object']).columns
for col in cat_cols:
    display(compu_df[col].value_counts().to_frame(name='Frecuencia'))

In [ ]:
# Histogramas de todas las columnas numéricas.
compu_df.hist(figsize=(16, 14), bins=20)
plt.tight_layout()
plt.show()


In [ ]:
# Gráficos de barras de variables categóricas
for col in compu_df.select_dtypes(include='object'):
    compu_df[col].value_counts().head(10).plot(
        kind='bar', figsize=(8, 4), title=col
    )
    plt.ylabel('Frecuencia')
    plt.xlabel(col)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


3. Dibuja un mapa de calor con la matriz de correlación para las variables numéricas del conjunto de datos.
* Identifica los pares de variables cuya correlación sea superior a 0.9 e imprímelos.
* Reflexiona sobre cuáles variables, de las que se imprimieron, representan de manera general la capacidad del hardware y mantenlas; elimina las demás por aportar información redundante.
* Incluye una breve justificación de tus decisiones.

In [ ]:
# Calcular la correlación entre variables numéricas.
corr = compu_df.corr(numeric_only=True)

# Mapa de calor.
plt.figure(figsize=(14, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Matriz de correlación')
plt.show()


In [ ]:
# Revisión de la parte superior de la matriz.
upper_triangle = corr.where(
    np.triu(np.ones(corr.shape), k=1).astype(bool)
)

pares_correlacionados = (
    upper_triangle.stack()
    .loc[lambda x: x > 0.9]
    .sort_values(ascending=False)
)

pares_correlacionados

In [ ]:
# Conservamos los indicadores generales de capacidad del hardware.
compu_df.drop(columns=['cpu_cores', 'ram_gb'], inplace=True)

***Justificación:** Conservamos **cpu_tier** y **gpu_tier**, porque representan directamente los niveles de rendimiento del procesador y la tarjeta gráfica. Eliminamos **cpu_cores** y **ram_gb** para reducir la redundancia entre las variables altamente correlacionadas.*


4. Para comenzar con la ingeniería de características, crea una copia del dataframe y asígnala a un nuevo objeto llamado `compu_transf`.
* Calcula cuántos años han pasado desde el lanzamiento de cada computadora y almacénalo en una nueva columna llamada `years_since_release`; luego, elimina la columna original.
* Utiliza `KBinsDiscretizer` para reemplazar la columna `vram_gb` en 4 bins ordinales basados en cuantiles.
* Imprime los valores que delimitan cada bin y haz un gráfico de barras para ver la cantidad de observaciones en cada uno, con el fin de entender cómo se agruparon los datos.

In [ ]:
# Copia para realizar las transformaciones.
compu_transf = compu_df.copy()

In [ ]:
# Calcular la antigüedad utilizando el año actual.
compu_transf['years_since_release'] = 2026 - compu_transf['release_year']

# Eliminamos el año de lanzamiento original.
compu_transf.drop(columns='release_year', inplace=True)


In [ ]:
# Creación de los 4 grupos basados en cuantiles.
kbins = KBinsDiscretizer(
    n_bins=4, encode='ordinal', strategy='quantile'
)

# Reemplazamos VRAM por los grupos ordinales.
compu_transf['vram_gb'] = kbins.fit_transform(
    compu_transf[['vram_gb']]
).astype(int)

# Límites de los intervalos.
print("Límites de los bins:", kbins.bin_edges_[0])

# Grafico de barras de la cantidad de registros por grupo.
compu_transf['vram_gb'].value_counts().sort_index().plot(kind='bar')
plt.title('Distribución de VRAM por bins')
plt.xlabel('Bin ordinal de VRAM')
plt.ylabel('Frecuencia')
plt.show()


5. Observa los histogramas del ejercicio 2. Notarás que en las variables `charger_watts` y `psu_watts` aparece una barra en 0. Analiza por qué ocurre esto y qué significa en relación con el tipo de dispositivo.
* Como estas variables son mutuamente excluyentes, combínalas en una nueva columna llamada `power_watts` que contenga la potencia correspondiente de cada dispositivo y, a continuación, haz un histograma para verificar que la distribución resultante es bimodal.
* Por último, borra las columnas originales `charger_watts` y `psu_watts`.

*   ***charger_watts:** representa la potencia del cargador, correspondiente a dispositivos como laptops.*
*   ***psu_watts** representa la potencia de la fuente de alimentación, correspondiente a equipos de escritorio.*

Se espera una distribución bimodal porque las laptops suelen utilizar cargadores de menor potencia, mientras que las desktops utilizan fuentes de alimentación de mayor potencia. La combinación permite conservar ambas informaciones en una sola variable.

In [ ]:
# Combinación de ambas columnas, tomando la potencia correspondiente.
compu_transf['power_watts'] = (
    compu_transf['charger_watts'] + compu_transf['psu_watts']
)

# Distribución combinada.
plt.figure(figsize=(8, 4))
sns.histplot(compu_transf['power_watts'], bins=30)
plt.title('Distribución de potencia')
plt.xlabel('Potencia (watts)')
plt.show()

# Eliminación de las columnas originales.
compu_transf.drop(columns=['charger_watts', 'psu_watts'], inplace=True)


*La distribución de **power_watts** presenta dos grupos principales de valores:*

* *El primero se concentra aproximadamente entre 40 W y 330 W, asociado con la potencia de los cargadores de laptops.*
* *El segundo se extiende aproximadamente entre 450 W y 850 W, correspondiente a la potencia de las fuentes de alimentación de computadoras de escritorio.*

*Esta distribución refleja la presencia de dos subpoblaciones de dispositivos con diferentes requerimientos de potencia.*

*Con algunos valores más altos de 1000 W o casi 1200 W, posiblemente pertencientes de computadoras de escritorio con fuentes de alimentación de mayor potencia.*


6. Para disminuir el sesgo de la variable `price`, crea tres transformadores: logaritmo, raíz cuadrada y Box - Cox.
* Aplica cada transformador a la variable price, dejando el resultado en variables temporales. El objetivo es comparar los efectos de cada transformación antes de decidir cuál aplicar de manera definitiva sobre las variables continuas del dataframe.
* De la variable original y de cada una de las tres transformaciones se debe mostrar:
  * Histograma: para observar la distribución de los datos.
  * Boxplot: para identificar posibles valores atípicos.
  * Q-Q plot: para evaluar la normalidad de la variable.
  * Skew (sesgo): para cuantificar la asimetría de la distribución.
  * Cantidad de outliers: para conocer cuántos valores extremos existen.
* En función de los resultados obtenidos al comparar las transformaciones, decide cuál logró el mejor efecto sobre la distribución de la variable y aplícala directamente en el dataframe, reemplazando las variables continuas: `weight_kg`, `power_watts` y `price`.

In [ ]:
# Definimos las tres transformaciones.
log_transf = FunctionTransformer(np.log1p)
sqrt_transf = FunctionTransformer(np.sqrt)
boxcox_transf = PowerTransformer(method='box-cox', standardize=False)

# Aplicamos cada transformación a price.
precio = compu_transf[['price']]

precios = {
    'Original': precio['price'].values,
    'Logaritmo': log_transf.transform(precio).values.ravel(),
    'Raíz cuadrada': sqrt_transf.transform(precio).values.ravel(),
    'Box-Cox': boxcox_transf.fit_transform(precio).ravel()
}

In [ ]:
# Comparación de las 4 distribuciones.
for nombre, datos in precios.items():

    # Sesgo y outliers usando IQR.
    q1, q3 = np.percentile(datos, [25, 75])
    iqr = q3 - q1
    outliers = ((datos < q1 - 1.5*iqr) |
                (datos > q3 + 1.5*iqr)).sum()

    print(f'{nombre}: Skew = {stats.skew(datos):.3f}, '
          f'Outliers = {outliers}')

    fig, ax = plt.subplots(1, 3, figsize=(15, 4))

    # Histograma.
    sns.histplot(datos, bins=30, ax=ax[0])
    ax[0].set_title('Histograma')

    # Boxplot.
    sns.boxplot(x=datos, ax=ax[1])
    ax[1].set_title('Boxplot')

    # Q-Q plot.
    stats.probplot(datos, dist='norm', plot=ax[2])
    ax[2].set_title('Q-Q plot')

    fig.suptitle(nombre)
    plt.tight_layout()
    plt.show()


* ***Box-Cox** ofrece los mejores resultados, ya que reduce prácticamente a cero la asimetría y presenta la menor cantidad de outliers entre las transformaciones evaluadas. También se espera que su Q-Q plot muestre una mejor aproximación a la normalidad, aunque esto debe verificarse visualmente.**


In [ ]:
# Aplicación de Box-Cox a las tres variables continuas.
for col in ['weight_kg', 'power_watts', 'price']:
    transformador = PowerTransformer(
        method='box-cox', standardize=False
    )

    compu_transf[col] = transformador.fit_transform(
        compu_transf[[col]]
    )


7. Para que todas las variables numéricas estén en la misma escala, aplica `MinMaxScaler` de sklearn a todas las columnas numéricas del dataframe, reemplazando las columnas originales.

In [ ]:
# Selección de todas las columnas numéricas.
num_cols = compu_transf.select_dtypes(include='number').columns

# Escalamos y reemplazamos los valores originales.
scaler = MinMaxScaler()
compu_transf[num_cols] = scaler.fit_transform(
    compu_transf[num_cols]
)

# Verificación de los valores mínimos y máximos.
print(compu_transf[num_cols].agg(['min', 'max']))


8. Aunque `wifi` es una variable categórica, sus categorías tienen un orden natural (Wi-Fi 5 < Wi-Fi 6 < Wi-Fi 6E < Wi-Fi 7). Codifícala usando `OrdinalEncoder`.
* Luego, escala la variable codificada entre 0 y 1 con `MinMaxScaler`, para que quede en la misma escala que las variables numéricas del dataframe.

Nota: Ambos cambios deben efectuarse sobre la columna original, de manera que quede una única columna `wifi` con toda la información transformada.

In [ ]:
# Definición del orden de las categorías.
orden_wifi = [['Wi-Fi 5', 'Wi-Fi 6', 'Wi-Fi 6E', 'Wi-Fi 7']]

# Conversión de las categorías en números ordinales.
encoder = OrdinalEncoder(categories=orden_wifi)
compu_transf[['wifi']] = encoder.fit_transform(
    compu_transf[['wifi']]
)

# Escalamos la misma columna entre 0 y 1.
compu_transf[['wifi']] = MinMaxScaler().fit_transform(
    compu_transf[['wifi']]
)

# Revisión de los valores resultantes.
print(compu_transf['wifi'].value_counts().sort_index())


9. La variable `gpu_model` tiene muchas categorías. Usar *One-Hot Encoding* aumentaría significativamente la dimensionalidad del dataframe. Por ello, utiliza `BinaryEncoder` para codificarla.
* Guarda el resultado en un dataframe llamado `bin_df`. Más adelante, lo combinarás con `compu_transf` para integrar las variables codificadas.

In [ ]:
# Codificamos los modelos de GPU en formato binario.
binary = BinaryEncoder(cols=['gpu_model'])

# Guardamos las nuevas columnas en bin_df.
bin_df = binary.fit_transform(compu_transf[['gpu_model']])

# Revisión del resultado.
display(bin_df.head())
print("Columnas generadas:", bin_df.shape[1])


10. Usa `OneHotEncoder` para codificar las variables categóricas restantes. Asegúrate de usar `drop='first'` para evitar la multicolinealidad y guarda el resultado en un dataframe llamado `ohe_df`
* Combina el dataframe `compu_transf` con las variables categóricas que fueron codificadas en `bin_df` y `ohe_df`. No olvides eliminar las variables originales.
* Usa `describe()` sobre el dataframe resultante para corroborar que todas las columnas estén escaladas entre 0 y 1 y que no queden variables categóricas sin codificar.

In [ ]:
# Selección de las columnas de texto, excepto gpu_model.
cat_cols = compu_transf.select_dtypes(include='object').columns.drop('gpu_model')

# Creación del codificador eliminando la primera categoría.
ohe = OneHotEncoder(drop='first', sparse_output=False)

# Codificación de las variables categóricas.
datos_ohe = ohe.fit_transform(compu_transf[cat_cols])

# Creación del DataFrame con las nuevas columnas.
ohe_df = pd.DataFrame(
    datos_ohe,
    columns=ohe.get_feature_names_out(cat_cols),
    index=compu_transf.index
)

display(ohe_df.head())


In [ ]:
# Eliminación de las columnas categóricas originales.
compu_transf.drop(
    columns=list(cat_cols) + ['gpu_model'],
    inplace=True
)

# Incorporamos las columnas codificadas.
compu_transf = pd.concat(
    [compu_transf, bin_df, ohe_df],
    axis=1
)

# Revisamos el resultado.
print("Dimensiones finales:", compu_transf.shape)
display(compu_transf.head())


In [ ]:
# Estadísticas descriptivas finales.
display(compu_transf.describe())

# Comprobación de que todas las columnas sean numéricas.
print("Columnas de texto:",
      len(compu_transf.select_dtypes(include='object').columns))

# Verificación final
print("Valor mínimo:", compu_transf.min().min())
print("Valor máximo:", compu_transf.max().max())
print("Valores faltantes:", compu_transf.isnull().sum().sum())


---
Google. (2026). Gemini (basado en Gemini 2.5) [Modelo de lenguaje grande], utilizado para la depuración y optimización del código. https://gemini.google.com/
---